In [0]:
import sys
import os

# Get the current notebook path dynamically
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Convert to workspace path and go up two levels (notebook -> bronze -> Air quality)
config_dir = '/Workspace' + os.path.dirname(os.path.dirname(notebook_path))
sys.path.append(config_dir)

from config import OpenAQ_API_KEY
import requests
import json
import datetime

In [0]:
df = spark.read.json('/Volumes/openaq/bronze_dev/locations/locations_2026-07-132026-07-13.json')
result_list = df.select("sensors", "id").toPandas().to_dict(orient='records')

In [0]:
for item in result_list:
    location_id = item.get('id')
    sensors = item.get('sensors', [])
    for sensor in sensors:
        sensor_id = sensor.get('id')
        url = f'https://api.openaq.org/v3/sensors/{sensor_id}/measurements/hourly?datetime_to=2026-07-13T11%3A19%3A38-06%3A00&datetime_from=2026-07-12T11%3A19%3A38-06%3A00&limit=100&page=1'
        response = requests.get(url, headers={'X-API-Key': OpenAQ_API_KEY})
        result = response.json().get('results')
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        dir_path = f'/Volumes/openaq/bronze_dev/sensor_data/{location_id}_{sensor_id}'
        os.makedirs(dir_path, exist_ok=True)
        with open(f'{dir_path}/{sensor_id}_{timestamp}.json', 'w') as f:
            f.write(json.dumps(result))

In [0]:
%sql
select * from read_files('/Volumes/openaq/bronze_dev/sensor_data/*')

In [0]:
#need to make sure I am properly working with pagination. so if limit is les than found go fetch all the pages
#save the data in folders of location_id + sensor_id, than inside of the folders set the timestamp as the name of the file or sensor_data+timestamp, it is now quite messy